# DeepSkin — EfficientNetB2 + CBAM (Local/Office Server Version)

**Architecture:** EfficientNetB2 (ImageNet pretrained) → CBAM Attention → GAP → BN → Dropout(0.5) → Dense(sigmoid)

**Training:**
- Phase 1: Head + CBAM only (backbone frozen) — BinaryCrossentropy + class_weight
- Phase 2: + block7 SE gates (conservative unfreeze) — BinaryCrossentropy + class_weight

> **Before running:** put your dataset in `../assets/` next to this notebook, with `benign/` and `malignant/` subfolders (directly, or one level down inside assets).


## 1. GPU Check

In [1]:
import tensorflow as tf
import os

print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU devices:', gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU memory growth enabled.')

    # ── Mixed Precision (set BEFORE any model/layer is built) ──
    # Uses float16 for most compute (much faster on RTX 5090 Tensor Cores),
    # keeps weights/accumulation in float32 for numerical stability.
    # Keras's model.compile()/fit() automatically applies dynamic loss-scaling
    # under this policy -- no manual LossScaleOptimizer needed for the
    # standard .fit() API used in this notebook.
    # tf.keras.mixed_precision.set_global_policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy('float32')
    print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())
else:
    print('WARNING: No GPU found. Enable GPU in Accelerator settings.')


I0000 00:00:1783267059.947671  374168 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783267060.213051  374168 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783267061.501441  374168 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TF version: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU memory growth enabled.
Mixed precision policy: <DTypePolicy "float32">


W0000 00:00:1783267063.125019  374168 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


## 2. Dataset Discovery (local assets/ folder)


In [2]:
import os
import glob

# ── Local dataset discovery (assets/ folder next to this notebook) ──
# Expects a PRE-SPLIT, deduplicated dataset:
#   assets/<...>/train/benign, assets/<...>/train/malignant
#   assets/<...>/val/benign,   assets/<...>/val/malignant
# (produced by dedup_and_split.py — see split_manifest.csv for provenance)
DATA_ROOT = os.path.join(os.getcwd(), '../assets')
print(f'Looking for dataset under: {DATA_ROOT}')

train_benign_hits = glob.glob(os.path.join(DATA_ROOT, '**', 'train', 'benign'), recursive=True)
if not train_benign_hits:
    raise FileNotFoundError(
        f"Could not find a 'train/benign' folder under {DATA_ROOT}. "
        f"Expected a pre-split structure like assets/<name>/train/benign, "
        f"assets/<name>/train/malignant, assets/<name>/val/benign, assets/<name>/val/malignant. "
        f"Run dedup_and_split.py first if you haven't already."
    )
train_dir = os.path.dirname(train_benign_hits[0])
val_dir   = os.path.join(os.path.dirname(train_dir), 'val')
if not os.path.isdir(os.path.join(val_dir, 'benign')):
    raise FileNotFoundError(f"Found train_dir={train_dir} but no matching val_dir at {val_dir}")

print(f'train_dir: {train_dir}')
print(f'val_dir:   {val_dir}')

num_benign_train    = len(os.listdir(os.path.join(train_dir, 'benign')))
num_malignant_train = len(os.listdir(os.path.join(train_dir, 'malignant')))
num_benign_val       = len(os.listdir(os.path.join(val_dir, 'benign')))
num_malignant_val    = len(os.listdir(os.path.join(val_dir, 'malignant')))

total_train = num_benign_train + num_malignant_train
total_val   = num_benign_val + num_malignant_val

print(f'Train — Benign: {num_benign_train}  Malignant: {num_malignant_train}  '
      f'Ratio: {num_benign_train/num_malignant_train:.2f}:1')
print(f'Val   — Benign: {num_benign_val}  Malignant: {num_malignant_val}  '
      f'Ratio: {num_benign_val/num_malignant_val:.2f}:1')

# ── Output / checkpoint paths (local, next to the notebook) ──
PROJECT_DIR    = os.path.join(os.getcwd(), 'DeepSkin_Project_CBAM')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
MODEL_FILE     = os.path.join(CHECKPOINT_DIR, 'efficientnet_b2_cbam_best.keras')
METADATA_FILE  = os.path.join(CHECKPOINT_DIR, 'training_metadata.json')
HISTORY_FILE   = os.path.join(PROJECT_DIR,    'training_history.json')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PROJECT_DIR,    exist_ok=True)
print(f'\nCheckpoints: {CHECKPOINT_DIR}')


Looking for dataset under: /home/higainai/project/deepskin_v4_gpu_optimization/../assets
train_dir: /home/higainai/project/deepskin_v4_gpu_optimization/../assets/DeepSkin_Data_Clean/train
val_dir:   /home/higainai/project/deepskin_v4_gpu_optimization/../assets/DeepSkin_Data_Clean/val
Train — Benign: 12792  Malignant: 7472  Ratio: 1.71:1
Val   — Benign: 3199  Malignant: 1868  Ratio: 1.71:1

Checkpoints: /home/higainai/project/deepskin_v4_gpu_optimization/DeepSkin_Project_CBAM/checkpoints


## 3. (Optional) Resume From an External Checkpoint Backup

If you're resuming from a checkpoint you copied in from elsewhere (e.g. another
machine), drop the files into `EXTERNAL_CHECKPOINT_DIR` below and run this cell.
Otherwise it's a no-op — safe to just run and move on.


In [3]:
import shutil

# Point this at a folder of externally-saved checkpoints if you have one.
EXTERNAL_CHECKPOINT_DIR = os.path.join(os.getcwd(), 'external_checkpoints')
if os.path.exists(EXTERNAL_CHECKPOINT_DIR):
    for fname in os.listdir(EXTERNAL_CHECKPOINT_DIR):
        src = os.path.join(EXTERNAL_CHECKPOINT_DIR, fname)
        dst = os.path.join(CHECKPOINT_DIR, fname)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f'Copied: {fname}')
    print('Checkpoint copy done.')
else:
    print('No external checkpoint folder found — starting fresh (normal for first run).')


No external checkpoint folder found — starting fresh (normal for first run).


## 4. Imports

In [4]:
import tensorflow as tf
import numpy as np
import os, json, time, glob
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    recall_score, precision_score, fbeta_score,
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    accuracy_score, f1_score, roc_auc_score
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras import layers

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

TF version: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 5. Session Timer

Kept mostly for logging/progress purposes. Since this is running on your own
server (no Kaggle 9-hour session limit), the ceiling below is set very high so
it will never cut off a real training run.


In [5]:
SESSION_START_TIME  = time.time()
# No Kaggle session limit here — this is just for progress logging.
MAX_SESSION_HOURS   = 1000
MAX_SESSION_SECONDS = MAX_SESSION_HOURS * 3600

def time_remaining():
    remaining = MAX_SESSION_SECONDS - (time.time() - SESSION_START_TIME)
    h = int(remaining // 3600)
    m = int((remaining % 3600) // 60)
    return f'{h}h {m}m remaining (informational only)'

def session_nearly_over():
    return (time.time() - SESSION_START_TIME) > MAX_SESSION_SECONDS

print('Session timer started (local run, no hard limit).')


Session timer started (local run, no hard limit).


## 6. Focal Loss

**alpha=0.35** matches the 2.13:1 class ratio (mathematically recommended = 0.32).  
Higher alpha values (0.75, 0.90) caused model collapse in previous runs.

In [6]:
ALPHA_VALUE  = 0.35   # matches 2.13:1 class ratio
GAMMA_PHASE1 = 1.0    # gentler for Phase 1 head training
GAMMA_PHASE2 = 2.0    # standard focal loss for Phase 2 fine-tuning

class BinaryFocalLoss(tf.keras.losses.Loss):
    def __init__(self, alpha=0.35, gamma=1.0, name='binary_focal_loss'):
        super().__init__(name=name)
        self.alpha = alpha
        self.gamma = gamma

    def call(self, y_true, y_pred):
        # Cast to float32 BEFORE computation — prevents float16 overflow
        y_pred  = tf.cast(y_pred, tf.float32)
        y_true  = tf.cast(y_true, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce     = -(y_true * tf.math.log(y_pred) +
                    (1 - y_true) * tf.math.log(1 - y_pred))
        p_t     = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        focal_w = alpha_t * tf.pow(1.0 - p_t, self.gamma)
        return tf.reduce_mean(focal_w * bce)

    def get_config(self):
        return {'alpha': self.alpha, 'gamma': self.gamma, 'name': self.name}

print(f'BinaryFocalLoss ready.')
print(f'  Phase 1: alpha={ALPHA_VALUE}, gamma={GAMMA_PHASE1}')
print(f'  Phase 2: alpha={ALPHA_VALUE}, gamma={GAMMA_PHASE2}')

BinaryFocalLoss ready.
  Phase 1: alpha=0.35, gamma=1.0
  Phase 2: alpha=0.35, gamma=2.0


## 7. CBAM Attention Module

```
Input Feature Map
      │
      ▼
Channel Attention  ← GlobalAvgPool + GlobalMaxPool → shared MLP → sigmoid → scale
      │
      ▼
Spatial Attention  ← AvgPool + MaxPool (channel-wise) → Conv7x7 → sigmoid → scale
      │
      ▼
Refined Feature Map  (lesion highlighted, background suppressed)
```

In [7]:
class ChannelAttention(layers.Layer):
    def __init__(self, reduction_ratio=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        channels = input_shape[-1]
        reduced  = max(1, channels // self.reduction_ratio)
        self.dense1 = layers.Dense(reduced,   activation='relu', use_bias=False)
        self.dense2 = layers.Dense(channels,  activation=None,   use_bias=False)
        super().build(input_shape)

    def call(self, x):
        avg   = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        avg   = self.dense2(self.dense1(avg))
        mx    = tf.reduce_max(x,  axis=[1, 2], keepdims=True)
        mx    = self.dense2(self.dense1(mx))
        scale = tf.sigmoid(avg + mx)
        return x * scale

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction_ratio': self.reduction_ratio})
        return cfg


class SpatialAttention(layers.Layer):
    def __init__(self, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size
        self.conv = layers.Conv2D(
            filters=1, kernel_size=kernel_size,
            padding='same', activation='sigmoid', use_bias=False
        )

    def call(self, x):
        avg      = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx       = tf.reduce_max(x,  axis=-1, keepdims=True)
        combined = tf.concat([avg, mx], axis=-1)
        return x * self.conv(combined)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'kernel_size': self.kernel_size})
        return cfg


class CBAM(layers.Layer):
    def __init__(self, reduction_ratio=16, kernel_size=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio
        self.kernel_size     = kernel_size
        self.channel_att     = ChannelAttention(reduction_ratio)
        self.spatial_att     = SpatialAttention(kernel_size)

    def call(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction_ratio': self.reduction_ratio,
                    'kernel_size': self.kernel_size})
        return cfg


CUSTOM_OBJECTS = {
    'BinaryFocalLoss' : BinaryFocalLoss,
    'CBAM'            : CBAM,
    'ChannelAttention': ChannelAttention,
    'SpatialAttention': SpatialAttention,
}
print('CBAM defined: ChannelAttention + SpatialAttention + CBAM')

CBAM defined: ChannelAttention + SpatialAttention + CBAM


## 8. Build Model

In [8]:
def build_model_with_cbam():
    """
    EfficientNetB2 (frozen) → CBAM → GAP → BN → Dropout(0.5) → Dense(sigmoid)
    Proposal spec: Dropout=0.5, input_shape=(260,260,3)

    NOTE: output layer is forced to dtype='float32' even though the global
    policy is mixed_float16 -- this keeps the final sigmoid + loss computation
    in float32 for numerical stability, per Keras mixed-precision guidance.
    """
    from tensorflow.keras.applications import EfficientNetB2
    from tensorflow.keras.layers import (
        GlobalAveragePooling2D, Dropout, Dense, BatchNormalization
    )
    from tensorflow.keras.models import Model

    base = EfficientNetB2(
        weights='imagenet',
        include_top=False,
        input_shape=(260, 260, 3),
        name='efficientnetb2'
    )
    base.trainable = False

    x = CBAM(name='cbam')(base.output)
    x = GlobalAveragePooling2D(name='gap')(x)
    x = BatchNormalization(name='bn_head')(x)
    x = Dropout(0.5, name='dropout_head')(x)
    out = Dense(1, activation='sigmoid', name='output', dtype='float32')(x)

    return Model(inputs=base.input, outputs=out, name='DeepSkin_CBAM'), base

_tmp, _ = build_model_with_cbam()
print(f'Model: DeepSkin_CBAM')
print(f'Total params: {_tmp.count_params():,}')
print(f'Output layer dtype: {_tmp.get_layer("output").dtype}  (should be float32)')

# Verify top_activation index (needed for Phase 2)
TOP_ACT_IDX = next(i for i, l in enumerate(_tmp.layers) if l.name == 'top_activation')
print(f'top_activation index: {TOP_ACT_IDX}')
del _tmp


W0000 00:00:1783267063.315673  374168 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


I0000 00:00:1783267063.392705  374168 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 29234 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:01:00.0, compute capability: 12.0a


Model: DeepSkin_CBAM
Total params: 8,023,516
Output layer dtype: float32  (should be float32)
top_activation index: 339


## 9. Data Generators

In [9]:
# ── Optimized tf.data input pipeline (replaces ImageDataGenerator) ──
# Why: ImageDataGenerator does file I/O + augmentation single-threaded on the
# CPU, in lockstep with the training loop -- this is almost always the
# bottleneck behind low GPU utilization (e.g. 6% core usage), since the GPU
# sits idle waiting for each batch. tf.data with parallel .map() + .prefetch()
# overlaps CPU preprocessing with GPU compute, and .cache() avoids re-decoding
# JPEGs from disk every epoch once the dataset fits in RAM.

AUTOTUNE   = tf.data.AUTOTUNE
BATCH_SIZE = 256     # scaled up from 32 -- see note below on VRAM headroom
IMG_SIZE   = (260, 260)

# class_names fixed and sorted so label indices match the original
# flow_from_directory behavior (alphabetical: benign=0, malignant=1)
class_names = sorted(os.listdir(train_dir))
print('Class names (index order):', class_names)

# GPU-acceleratable augmentation stack (Keras preprocessing layers).
# Each layer applies independent random parameters per-example within a
# batch (not one shared random draw for the whole batch), matching what
# ImageDataGenerator did per-image. Mirrors the original augmentation:
# rotation ±30°, width/height shift ±10%, zoom ±15%, h/v flip, brightness.
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(30 / 360),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
], name='data_augmentation')

def make_dataset(directory, training):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='binary',
        class_names=class_names,
        image_size=IMG_SIZE,
        batch_size=None,
        shuffle=training,
        seed=42,
    )
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y),
                num_parallel_calls=AUTOTUNE)

    n_samples = sum(1 for _ in ds)

    ds = ds.cache()  # keep cache — it's correct for both train and val

    if training:
        ds = ds.shuffle(2000, seed=42, reshuffle_each_iteration=True)

    ds = ds.batch(BATCH_SIZE)

    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)

    # FIX: limit prefetch to 2 — AUTOTUNE was requesting 200MB+
    # causing the prefetch autotuner warning and potential memory pressure
    ds = ds.prefetch(2)  # changed from AUTOTUNE
    return ds, n_samples


train_dataset, n_train = make_dataset(train_dir, training=True)
val_dataset,   n_val   = make_dataset(val_dir,   training=False)


class DatasetMetadata:
    """
    Lightweight shim that exposes .class_indices, .classes, and __len__
    to match the ImageDataGenerator interface used by callbacks and
    evaluation cells.
    Does NOT wrap the actual data — model.fit()/predict() calls use
    train_dataset/val_dataset directly, not this object.
    """
    def __init__(self, dataset, directory, n_samples, batch_size, class_names):
        self.directory    = directory
        self.target_size  = IMG_SIZE
        self.n_samples    = n_samples
        self.batch_size   = batch_size
        self.class_indices = {name: i for i, name in enumerate(class_names)}
        self._classes     = None
        self._dataset_for_labels = dataset

    def __len__(self):
        return int(np.ceil(self.n_samples / self.batch_size))

    def reset(self):
        pass  # tf.data.Dataset re-iterates cleanly — no state to reset

    @property
    def classes(self):
        if self._classes is None:
            # FIX: Build from filesystem — avoids tensor conversion issues
            # with mixed precision (float16) and 0-d tensor scalars in TF 2.21
            class_to_idx = self.class_indices  # {'benign': 0, 'malignant': 1}
            labels = []
            for class_name, idx in sorted(class_to_idx.items(), key=lambda x: x[1]):
                class_dir = os.path.join(self.directory, class_name)
                if os.path.isdir(class_dir):
                    n_files = len([
                        f for f in os.listdir(class_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                    ])
                    labels.extend([idx] * n_files)
            self._classes = np.array(labels, dtype=np.int32)
            print(f'  [DatasetMetadata] classes built: '
                  f'{(self._classes == 0).sum()} benign, '
                  f'{(self._classes == 1).sum()} malignant')
        return self._classes


train_generator      = DatasetMetadata(train_dataset, train_dir, n_train, BATCH_SIZE, class_names)
validation_generator = DatasetMetadata(val_dataset,   val_dir,   n_val,   BATCH_SIZE, class_names)

# Class weights (computed from TRAIN split only)
num_b = len(os.listdir(os.path.join(train_dir, 'benign')))
num_m = len(os.listdir(os.path.join(train_dir, 'malignant')))
tot   = num_b + num_m
class_weight_dict = {0: tot / (2 * num_b), 1: tot / (2 * num_m)}

print(f'Class indices: {train_generator.class_indices}')
print(f'Train samples: {n_train}  ({len(train_generator)} batches of {BATCH_SIZE})')
print(f'Val samples:   {n_val}  ({len(validation_generator)} batches of {BATCH_SIZE})')
print(f'Class weights: {class_weight_dict}')

Class names (index order): ['benign', 'malignant']
Found 20264 files belonging to 2 classes.


Found 5067 files belonging to 2 classes.
Class indices: {'benign': 0, 'malignant': 1}
Train samples: 20264  (80 batches of 256)
Val samples:   5067  (20 batches of 256)
Class weights: {0: 0.792057535959975, 1: 1.3559957173447537}


## 10. Callbacks

In [10]:
class RealRecallCallback(tf.keras.callbacks.Callback):
    """Reports true recall/precision/F2 at multiple thresholds.
    Workaround for Keras Recall metric returning 0.0 with focal loss.
    Takes the raw tf.data.Dataset + a precomputed labels array (via the
    DatasetMetadata shim's .classes), since tf.data.Dataset has no .reset()."""
    def __init__(self, val_dataset, val_labels):
        super().__init__()
        self.val_dataset = val_dataset
        self.val_labels  = val_labels

    def on_epoch_end(self, epoch, logs=None):
        preds  = self.model.predict(self.val_dataset, verbose=0).flatten()
        labels = self.val_labels
        print(f'\n  [RealRecall] Epoch {epoch+1}:')
        for thresh in [0.3, 0.4, 0.5]:
            y_t = (preds >= thresh).astype(int)
            r   = recall_score(labels, y_t, zero_division=0)
            p   = precision_score(labels, y_t, zero_division=0)
            f2  = fbeta_score(labels, y_t, beta=2, zero_division=0)
            print(f'    thresh={thresh}: recall={r:.4f}  '
                  f'precision={p:.4f}  F2={f2:.4f}')


class FullModelCheckpoint(tf.keras.callbacks.Callback):
    """Saves best model (val_auc_pr) + latest epoch for reliable resume."""
    def __init__(self, model_file, meta_file, phase, current_epoch=0):
        super().__init__()
        self.model_file    = model_file
        self.meta_file     = meta_file
        self.phase         = phase
        self.current_epoch = current_epoch
        self.best_metric   = float('-inf')
        self.latest_file   = model_file.replace('.keras', '_latest.keras')
        self.latest_meta   = meta_file.replace('.json',  '_latest.json')

    def on_epoch_end(self, epoch, logs=None):
        self.current_epoch = epoch + 1
        metric = logs.get('val_auc_pr', float('-inf'))

        # Always save latest (for resume after timeout)
        try:
            self.model.save(self.latest_file)
            with open(self.latest_meta, 'w') as f:
                json.dump({'phase': self.phase, 'epoch': self.current_epoch,
                           'val_auc_pr': float(metric)}, f)
        except Exception as e:
            print(f'  Latest save failed: {e}')

        # Save best model
        if metric > self.best_metric:
            self.best_metric = metric
            try:
                self.model.save(self.model_file)
                with open(self.meta_file, 'w') as f:
                    json.dump({'phase': self.phase, 'epoch': self.current_epoch,
                               'val_auc_pr': float(metric)}, f)
                print(f'  ✓ Best saved — epoch {self.current_epoch}, '
                      f'val_auc_pr={metric:.4f}')
            except Exception as e:
                print(f'  Best save failed: {e}')
        else:
            print(f'  No improvement (best={self.best_metric:.4f})')


class SessionTimeoutCallback(tf.keras.callbacks.Callback):
    """Stops training safely before Kaggle session expires."""
    def on_epoch_end(self, epoch, logs=None):
        print(f'  {time_remaining()}')
        if session_nearly_over():
            print('WARNING: Session nearly over — stopping training safely.')
            self.model.stop_training = True


tracking_metrics = [
    'accuracy',
    Precision(name='precision', thresholds=0.5),
    Recall(name='recall',       thresholds=0.5),
    AUC(name='auc_roc', curve='ROC'),
    AUC(name='auc_pr',  curve='PR'),
]

base_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc_pr', patience=8, mode='max',
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc_pr', factor=0.5, patience=4,
        mode='max', verbose=1
    ),
]

real_recall_cb = RealRecallCallback(val_dataset, validation_generator.classes)
print('Callbacks ready.')


  [DatasetMetadata] classes built: 3199 benign, 1868 malignant
Callbacks ready.


## 11. Load Checkpoint or Start Fresh

In [11]:
# Prefer latest checkpoint (saved every epoch) over best (saved on improvement)
LATEST_MODEL = MODEL_FILE.replace('.keras', '_latest.keras')
LATEST_META  = METADATA_FILE.replace('.json',  '_latest.json')

RESUME_MODEL = LATEST_MODEL if os.path.exists(LATEST_MODEL) else MODEL_FILE
RESUME_META  = LATEST_META  if os.path.exists(LATEST_META)  else METADATA_FILE

if os.path.exists(RESUME_MODEL) and os.path.exists(RESUME_META):
    print('Checkpoint found — loading...')
    model = tf.keras.models.load_model(
        RESUME_MODEL, custom_objects=CUSTOM_OBJECTS, compile=False
    )
    with open(RESUME_META, 'r') as f:
        meta = json.load(f)
    start_phase = meta['phase']
    start_epoch = meta['epoch']
    print(f'Resumed: phase={start_phase}, epoch={start_epoch}, '
          f'val_auc_pr={meta["val_auc_pr"]:.4f}')
    if start_phase == 'complete':
        print('Training already complete. Run evaluation cells below.')
else:
    print('No checkpoint — starting fresh.')
    model, _ = build_model_with_cbam()
    start_phase = 'phase1'
    start_epoch = 0
    print(f'start_phase={start_phase}, start_epoch={start_epoch}')

Checkpoint found — loading...


E0000 00:00:1783267067.889877  374168 util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
/home/higainai/project/venv/lib/python3.12/site-packages/keras/src/layers/layer.py:431: UserWarning: `build()` was called on layer 'cbam', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Resumed: phase=phase2, epoch=0, val_auc_pr=0.0000


## 12. Phase 1 — Train Head + CBAM

Backbone frozen. BinaryCrossentropy + class_weight.  
Learning rate: 1e-3 (proposal spec).  
Augmentation: rotation, shift, zoom, flip, brightness via ImageDataGenerator.

In [12]:
if start_phase == 'phase1':
    print('=== Phase 1: Training Head + CBAM (backbone frozen) ===')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.BinaryCrossentropy(dtype='float32'),
        metrics=tracking_metrics,
        jit_compile=True   # XLA: fuses ops, reduces memory-transfer overhead
    )

    trainable = sum([tf.size(w).numpy() for w in model.trainable_weights])
    print(f'Trainable params: {trainable:,}  (CBAM + head only)')

    phase1_ckpt = FullModelCheckpoint(
        MODEL_FILE, METADATA_FILE, 'phase1', start_epoch
    )

    history_phase1 = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=10,
        initial_epoch=start_epoch,
        class_weight=class_weight_dict,
        callbacks=base_callbacks + [
            phase1_ckpt, real_recall_cb, SessionTimeoutCallback()
        ]
    )

    # Save Phase 1 backup before transitioning
    phase1_backup = os.path.join(PROJECT_DIR, 'phase1_complete_model.keras')
    model.save(phase1_backup)
    print(f'Phase 1 backup: {phase1_backup}')

    with open(METADATA_FILE, 'w') as f:
        json.dump({'phase': 'phase2', 'epoch': 0, 'val_auc_pr': 0.0}, f)

    actual = len(history_phase1.history['loss'])
    print(f'Phase 1 complete. Ran {actual} epochs.')
    start_phase = 'phase2'
    start_epoch = 0

else:
    print(f'Skipping Phase 1 (start_phase={start_phase})')


Skipping Phase 1 (start_phase=phase2)


## 13. Phase 2 — Fine-tune Block7 SE Gates + CBAM

**Conservative unfreeze:** Only block7a/b SE squeeze-excite + dwconv layers (~786K params).  
Massive expand/project convs (743K each) stay frozen to prevent catastrophic forgetting.  
All BatchNorm layers frozen for stability.  
Safety check: raises error if trainable params exceed 900K.

In [13]:
if start_phase == 'phase2':
    print('=== Phase 2: Fine-tuning Block7 SE Gates + CBAM ===')

    # Load Phase 1 backup cleanly to avoid corrupted weights
    phase1_backup = os.path.join(PROJECT_DIR, 'phase1_complete_model.keras')
    if os.path.exists(phase1_backup):
        model = tf.keras.models.load_model(
            phase1_backup, custom_objects=CUSTOM_OBJECTS, compile=False
        )
        print('Loaded Phase 1 backup cleanly.')
    else:
        print('Warning: No Phase 1 backup found — using current model.')

    # Step 1: Freeze ALL layers
    for layer in model.layers:
        layer.trainable = False

    # Step 2: Unfreeze ONLY conservative lightweight layers
    # block7 SE gates: small param layers that learn channel attention
    # Avoids unfreezing massive expand/project convs (743K params each)
    UNFREEZE_NAMES = {
        'block7b_dwconv',    # 19,008 params
        'block7b_se_reduce', # 185,944 params
        'block7b_se_expand', # 187,968 params
        'block7a_dwconv',    # 19,008 params
        'block7a_se_reduce', # 185,944 params
        'block7a_se_expand', # 187,968 params
        # Head (already trained Phase 1, continue refining)
        'cbam',
        'gap',
        'bn_head',
        'dropout_head',
        'output',
    }
    for layer in model.layers:
        if layer.name in UNFREEZE_NAMES:
            layer.trainable = True

    # Step 3: Keep ALL BatchNorm frozen for training stability
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    # Verify counts
    trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
    trainable_names  = [l.name for l in model.layers if l.trainable]
    bn_frozen        = sum(1 for l in model.layers
                           if isinstance(l, tf.keras.layers.BatchNormalization))

    print(f'Trainable params: {trainable_params:,}')
    print(f'BatchNorm frozen: {bn_frozen}')
    print(f'Trainable layers: {trainable_names}')

    # Safety check — stop before collapse
    if trainable_params > 900_000:
        raise ValueError(
            f'Too many trainable params ({trainable_params:,}). '
            f'Expected ~786,000. Check UNFREEZE_NAMES list.'
        )
    print(f'Param count OK ({trainable_params:,} < 900,000). Proceeding.')

    # Compile: binary_crossentropy safer than focal loss for fine-tuning
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=tracking_metrics,
        jit_compile=True   # XLA: fuses ops, reduces memory-transfer overhead
    )

    phase2_ckpt = FullModelCheckpoint(
        MODEL_FILE, METADATA_FILE, 'phase2', start_epoch
    )

    PHASE1_EPOCHS    = 10
    PHASE2_MAX_EPOCH = 50  # total budget (absolute epoch count, not relative)

    history_phase2 = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=PHASE2_MAX_EPOCH,   # FIXED: absolute target epoch, not a relative count
                                   # (model.fit's `epochs` + `initial_epoch` together define
                                   # an absolute range; using a relative count here previously
                                   # caused a resumed run to see initial_epoch >= epochs and
                                   # silently train for zero epochs)
        initial_epoch=start_epoch,
        class_weight=class_weight_dict,
        callbacks=base_callbacks + [
            phase2_ckpt, real_recall_cb, SessionTimeoutCallback()
        ]
    )

    if len(history_phase2.history.get('loss', [])) == 0:
        print(f'No epochs were run (initial_epoch={start_epoch} >= epochs={PHASE2_MAX_EPOCH}).')
        print('Training budget already exhausted for this phase.')
        actual = 0
    else:
        actual = len(history_phase2.history['loss'])
        print(f'Phase 2 ran for {actual} epochs.')

    final_path = os.path.join(PROJECT_DIR, 'final_model_cbam.keras')
    model.save(final_path)
    print(f'Final model: {final_path}')

    with open(METADATA_FILE, 'w') as f:
        json.dump({
            'phase': 'complete', 'epoch': actual,
            'val_auc_pr': float(max(history_phase2.history.get('val_auc_pr', [0])))
        }, f)

    start_phase = 'complete'

else:
    print(f'Skipping Phase 2 (start_phase={start_phase})')


=== Phase 2: Fine-tuning Block7 SE Gates + CBAM ===
Loaded Phase 1 backup cleanly.
Trainable params: 784,559
BatchNorm frozen: 70
Trainable layers: ['block7a_dwconv', 'block7a_se_reduce', 'block7a_se_expand', 'block7b_dwconv', 'block7b_se_reduce', 'block7b_se_expand', 'cbam', 'gap', 'dropout_head', 'output']
Param count OK (784,559 < 900,000). Proceeding.
Epoch 1/50


/home/higainai/project/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1783267076.927185  374234 service.cc:153] XLA service 0x7057b0085710 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1783267076.927249  374234 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5090, Compute Capability 12.0a (Driver: 13.2.0; Runtime: 12.6.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1783267077.849366  374234 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1783267079.671816  374234 cuda_dnn.cc:461] Loaded cuDNN version 91002
I0000 00:00:1783267080.021154  374234 dot_merger.cc:481] Merging

 3/80 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.8125 - auc_pr: 0.8239 - auc_roc: 0.8986 - loss: 0.4158 - precision: 0.7470 - recall: 0.6966 

I0000 00:00:1783267100.165565  374234 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


79/80 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.7892 - auc_pr: 0.7964 - auc_roc: 0.8724 - loss: 0.4453 - precision: 0.6904 - recall: 0.7757

...

80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 377ms/step - accuracy: 0.7891 - auc_pr: 0.7965 - auc_roc: 0.8723 - loss: 0.4455 - precision: 0.6907 - recall: 0.7754

I0000 00:00:1783267129.899515  374235 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_532', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1783267132.630152  374235 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_93347__.52
I0000 00:00:1783267137.692591  374231 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_93347__.52
E0000 00:00:1783267138.088076  374231 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1783267138.357307  374231 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1783267138.538540  374231 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup executio

  ✓ Best saved — epoch 1, val_auc_pr=0.7995


I0000 00:00:1783267147.869025  374235 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_101986__.29
I0000 00:00:1783267153.183816  374231 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_106820__.29



  [RealRecall] Epoch 1:
    thresh=0.3: recall=0.9293  precision=0.5614  F2=0.8217
    thresh=0.4: recall=0.8737  precision=0.6194  F2=0.8074
    thresh=0.5: recall=0.8019  precision=0.6754  F2=0.7730
  999h 58m remaining (informational only)
80/80 ━━━━━━━━━━━━━━━━━━━━ 86s 703ms/step - accuracy: 0.7891 - auc_pr: 0.7965 - auc_roc: 0.8723 - loss: 0.4455 - precision: 0.6907 - recall: 0.7754 - val_accuracy: 0.7849 - val_auc_pr: 0.7995 - val_auc_roc: 0.8711 - val_loss: 0.4515 - val_precision: 0.6754 - val_recall: 0.8019 - learning_rate: 1.0000e-05
Epoch 2/50
79/80 ━━━━━━━━━━━━━━━━━━━━ 0s 225ms/step - accuracy: 0.7858 - auc_pr: 0.8027 - auc_roc: 0.8743 - loss: 0.4402 - precision: 0.6783 - recall: 0.7974  ✓ Best saved — epoch 2, val_auc_pr=0.8016

  [RealRecall] Epoch 2:
    thresh=0.3: recall=0.9197  precision=0.5689  F2=0.8187
    thresh=0.4: recall=0.8667  precision=0.6265  F2=0.8050
    thresh=0.5: recall=0.8003  precision=0.6839  F2=0.7740
  999h 58m remaining (informational only)
80/80

## 14. Save Training History

In [14]:
HISTORY_KEYS = [
    'loss', 'val_loss', 'accuracy', 'val_accuracy',
    'recall', 'val_recall', 'precision', 'val_precision',
    'auc_roc', 'val_auc_roc', 'auc_pr', 'val_auc_pr'
]

try:
    hdata = {
        'phase1': {k: [float(v) for v in history_phase1.history.get(k, [])]
                   for k in HISTORY_KEYS},
        'phase2': {k: [float(v) for v in history_phase2.history.get(k, [])]
                   for k in HISTORY_KEYS},
    }
    with open(HISTORY_FILE, 'w') as f:
        json.dump(hdata, f)
    print(f'History saved: {HISTORY_FILE}')
except NameError:
    print('history_phase1/phase2 not in memory — skipping.')

history_phase1/phase2 not in memory — skipping.


## 15. Load Model & Generate Predictions

In [15]:
EVAL_MODEL_PATH = os.path.join(PROJECT_DIR, 'final_model_cbam.keras')
if not os.path.exists(EVAL_MODEL_PATH):
    EVAL_MODEL_PATH = MODEL_FILE
    print(f'Final model not found, using best checkpoint.')

print(f'Loading: {EVAL_MODEL_PATH}')
eval_model = tf.keras.models.load_model(
    EVAL_MODEL_PATH, custom_objects=CUSTOM_OBJECTS, compile=False
)
print('Model loaded.')

# val_dataset is a tf.data.Dataset -- no .reset()/steps needed, it re-iterates
# cleanly every call. Labels come from the DatasetMetadata shim (validation_generator.classes).
y_pred_proba = eval_model.predict(val_dataset, verbose=1).flatten()
y_true = validation_generator.classes
y_pred = (y_pred_proba >= 0.5).astype(int)

print(f'Pred range: [{y_pred_proba.min():.4f}, {y_pred_proba.max():.4f}]')
print(f'Pred mean:  {y_pred_proba.mean():.4f}')
print(f'Predicted malignant (>=0.5): {y_pred.sum()}')
print(f'Actual malignant:            {y_true.sum()}')


Loading: /home/higainai/project/deepskin_v4_gpu_optimization/DeepSkin_Project_CBAM/final_model_cbam.keras


/home/higainai/project/venv/lib/python3.12/site-packages/keras/src/layers/layer.py:431: UserWarning: `build()` was called on layer 'cbam', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model loaded.


I0000 00:00:1783268126.867990  374235 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_292913__.29


19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step

I0000 00:00:1783268133.096058  374232 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_297747__.29


20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 328ms/step
Pred range: [0.0002, 0.9944]
Pred mean:  0.4253
Predicted malignant (>=0.5): 2035
Actual malignant:            1868


## 16. Core Metrics

In [16]:
accuracy    = accuracy_score(y_true, y_pred)
precision   = precision_score(y_true, y_pred, zero_division=0)
recall      = recall_score(y_true, y_pred, zero_division=0)
f1          = f1_score(y_true, y_pred, zero_division=0)
f2          = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
auc_roc_val = roc_auc_score(y_true, y_pred_proba)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)  # = recall
gmean       = np.sqrt(sensitivity * specificity)
fnr_val     = fn / (fn + tp)
fpr_val     = fp / (fp + tn)

all_metrics = [
    ('Accuracy',    accuracy,    0.90, '>'),
    ('Precision',   precision,   0.85, '>'),
    ('Recall',      recall,      0.90, '>'),
    ('Specificity', specificity, 0.85, '>'),
    ('F1-Score',    f1,          0.87, '>'),
    ('F2-Score',    f2,          0.85, '>'),
    ('AUC-ROC',     auc_roc_val, 0.95, '>'),
    ('G-Mean',      gmean,       0.85, '>'),
    ('FNR',         fnr_val,     0.10, '<'),
    ('FPR',         fpr_val,     0.15, '<'),
]

print('=' * 58)
print('     DeepSkin + CBAM  EVALUATION RESULTS')
print('=' * 58)
print(f'{"Metric":<20} {"Value":>10} {"Target":>10} {"Status":>8}')
print('-' * 58)
passed = 0
for name, val, tgt, direction in all_metrics:
    ok = val > tgt if direction == '>' else val < tgt
    if ok: passed += 1
    print(f'{name:<20} {val:>10.4f} {tgt:>10.2f} {"OK" if ok else "--":>8}')
print('=' * 58)
print(f'Targets met: {passed}/{len(all_metrics)}')
print(f'\nConfusion Matrix: TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print('\nClassification Report:')
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=['Benign','Malignant']))

     DeepSkin + CBAM  EVALUATION RESULTS
Metric                    Value     Target   Status
----------------------------------------------------------
Accuracy                 0.7985       0.90       --
Precision                0.7081       0.85       --
Recall                   0.7714       0.90       --
Specificity              0.8143       0.85       --
F1-Score                 0.7384       0.87       --
F2-Score                 0.7579       0.85       --
AUC-ROC                  0.8797       0.95       --
G-Mean                   0.7926       0.85       --
FNR                      0.2286       0.10       --
FPR                      0.1857       0.15       --
Targets met: 0/10

Confusion Matrix: TP=1441  TN=2605  FP=594  FN=427

Classification Report:
              precision    recall  f1-score   support

      Benign       0.86      0.81      0.84      3199
   Malignant       0.71      0.77      0.74      1868

    accuracy                           0.80      5067
   macro avg    

## 17. Threshold Analysis

In [17]:
print('Threshold Sweep:')
print(f'{"Threshold":>10} {"Recall":>8} {"Precision":>10} {"F1":>8} {"F2":>8} {"Spec":>8}')
print('-' * 58)

best_f1_thresh = 0.5; best_f1 = 0.0
best_f2_thresh = 0.5; best_f2 = 0.0

for thresh in np.arange(0.1, 0.91, 0.05):
    y_t = (y_pred_proba >= thresh).astype(int)
    if y_t.sum() == 0: continue
    r   = recall_score(y_true, y_t, zero_division=0)
    p   = precision_score(y_true, y_t, zero_division=0)
    f1t = f1_score(y_true, y_t, zero_division=0)
    f2t = fbeta_score(y_true, y_t, beta=2, zero_division=0)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_true, y_t).ravel()
    sp  = tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0
    marks = []
    if f1t > best_f1: best_f1 = f1t; best_f1_thresh = thresh; marks.append('F1')
    if f2t > best_f2: best_f2 = f2t; best_f2_thresh = thresh; marks.append('F2')
    tag = f' <- best {" ".join(marks)}' if marks else ''
    print(f'{thresh:>10.2f} {r:>8.4f} {p:>10.4f} {f1t:>8.4f} {f2t:>8.4f} {sp:>8.4f}{tag}')

# Find threshold achieving recall >= 0.90
thresh90 = None
for t in np.linspace(0.01, 0.99, 1000):
    if recall_score(y_true, (y_pred_proba >= t).astype(int), zero_division=0) >= 0.90:
        thresh90 = t

print(f'\nBest F1 threshold : {best_f1_thresh:.2f}  (F1={best_f1:.4f})')
print(f'Best F2 threshold : {best_f2_thresh:.2f}  (F2={best_f2:.4f})')
if thresh90:
    r90 = recall_score(y_true, (y_pred_proba >= thresh90).astype(int), zero_division=0)
    p90 = precision_score(y_true, (y_pred_proba >= thresh90).astype(int), zero_division=0)
    print(f'Threshold for recall>=0.90: {thresh90:.4f} '
          f'(recall={r90:.4f}, precision={p90:.4f})')
else:
    print('Model cannot achieve recall>=0.90 at any threshold.')

Threshold Sweep:
 Threshold   Recall  Precision       F1       F2     Spec
----------------------------------------------------------
      0.10   0.9855     0.4670   0.6337   0.8065   0.3432 <- best F1 F2
      0.15   0.9706     0.4984   0.6586   0.8159   0.4295 <- best F1 F2
      0.20   0.9524     0.5314   0.6821   0.8221   0.5095 <- best F1 F2
      0.25   0.9208     0.5641   0.6996   0.8174   0.5846 <- best F1
      0.30   0.8999     0.5997   0.7198   0.8180   0.6493 <- best F1
      0.35   0.8758     0.6302   0.7330   0.8125   0.6999 <- best F1
      0.40   0.8464     0.6604   0.7419   0.8012   0.7459 <- best F1
      0.45   0.8116     0.6838   0.7422   0.7823   0.7809 <- best F1
      0.50   0.7714     0.7081   0.7384   0.7579   0.8143
      0.55   0.7270     0.7325   0.7297   0.7281   0.8450
      0.60   0.6847     0.7524   0.7169   0.6972   0.8684
      0.65   0.6354     0.7743   0.6980   0.6591   0.8918
      0.70   0.5948     0.8045   0.6839   0.6275   0.9156
      0.75   0.

## 18. Plots — Confusion Matrix, ROC, PR Curve

In [18]:
y_pred_f2 = (y_pred_proba >= best_f2_thresh).astype(int)

# ── Confusion Matrices ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Confusion Matrices — DeepSkin + CBAM', fontsize=14, fontweight='bold')

for ax, y_p, title, cmap in [
    (axes[0], y_pred,    'Threshold = 0.50 (default)', 'Blues'),
    (axes[1], y_pred_f2, f'Threshold = {best_f2_thresh:.2f} (best F2)', 'Oranges'),
]:
    cm = confusion_matrix(y_true, y_p)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=['Benign','Malignant'],
                yticklabels=['Benign','Malignant'], ax=ax)
    ax.set_title(title); ax.set_ylabel('True'); ax.set_xlabel('Predicted')

plt.tight_layout()
cm_path = os.path.join(PROJECT_DIR, 'confusion_matrices.png')
plt.savefig(cm_path, dpi=150); plt.show()
print(f'Saved: {cm_path}')

Saved: /home/higainai/project/deepskin_v4_gpu_optimization/DeepSkin_Project_CBAM/confusion_matrices.png


In [19]:
# ── ROC Curve ──
fpr_arr, tpr_arr, _ = roc_curve(y_true, y_pred_proba)
roc_auc_plot = auc(fpr_arr, tpr_arr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_arr, tpr_arr, 'darkorange', lw=2, label=f'AUC={roc_auc_plot:.4f}')
ax.plot([0,1],[0,1], 'navy', lw=1, linestyle='--', label='Random')
ax.scatter([fpr_val],[sensitivity], color='red', s=100, zorder=5,
           label=f'Default (recall={sensitivity:.3f})')
ax.axhline(y=0.90, color='green', linestyle=':', alpha=0.7, label='Target recall 0.90')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — DeepSkin + CBAM'); ax.legend(loc='lower right'); ax.grid(alpha=0.3)
roc_path = os.path.join(PROJECT_DIR, 'roc_curve.png')
plt.tight_layout(); plt.savefig(roc_path, dpi=150); plt.show()
print(f'AUC-ROC: {roc_auc_plot:.4f}  Saved: {roc_path}')

AUC-ROC: 0.8797  Saved: /home/higainai/project/deepskin_v4_gpu_optimization/DeepSkin_Project_CBAM/roc_curve.png


In [20]:
# ── Precision-Recall Curve ──
prec_c, rec_c, _ = precision_recall_curve(y_true, y_pred_proba)
avg_prec = average_precision_score(y_true, y_pred_proba)
baseline = y_true.sum() / len(y_true)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rec_c, prec_c, 'darkorange', lw=2, label=f'AP={avg_prec:.4f}')
ax.scatter([recall],[precision], color='red', s=100, zorder=5, label='Default thresh=0.50')
ax.scatter([recall_score(y_true, y_pred_f2)],
           [precision_score(y_true, y_pred_f2)],
           color='green', s=100, zorder=5, label=f'Best F2 thresh={best_f2_thresh:.2f}')
ax.axvline(x=0.90, color='purple', linestyle='--', alpha=0.7, label='Recall target')
ax.axhline(y=baseline, color='navy', linestyle='--', alpha=0.5,
           label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curve — DeepSkin + CBAM'); ax.legend(); ax.grid(alpha=0.3)
pr_path = os.path.join(PROJECT_DIR, 'pr_curve.png')
plt.tight_layout(); plt.savefig(pr_path, dpi=150); plt.show()
print(f'AP: {avg_prec:.4f}  Saved: {pr_path}')

AP: 0.8185  Saved: /home/higainai/project/deepskin_v4_gpu_optimization/DeepSkin_Project_CBAM/pr_curve.png


## 19. Training Curves

In [21]:
# Load history from file if not in memory
HIST_OK = False
try:
    _ = history_phase1; _ = history_phase2; HIST_OK = True
except NameError:
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as f:
            hd = json.load(f)
        class H:
            def __init__(self, d): self.history = d
        history_phase1 = H(hd['phase1'])
        history_phase2 = H(hd['phase2'])
        HIST_OK = True
        print('History loaded from file.')
    else:
        print('No history file. Skipping training curves.')

if HIST_OK:
    p1 = history_phase1.history
    p2 = history_phase2.history
    ep1 = len(p1.get('loss', []))
    ep2 = len(p2.get('loss', []))
    x1  = range(1, ep1+1)
    x2  = range(ep1+1, ep1+ep2+1)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('DeepSkin + CBAM Training History', fontsize=16, fontweight='bold')

    specs = [
        (axes[0,0], 'loss',     'Loss Curve',     None,  None),
        (axes[0,1], 'accuracy', 'Accuracy Curve', 0.90,  'Target 0.90'),
        (axes[1,0], 'recall',   'Recall Curve',   0.90,  'Target 0.90'),
        (axes[1,1], 'auc_pr',   'AUC-PR Curve',   0.95,  'Target 0.95'),
    ]

    for ax, key, title, tgt, tlbl in specs:
        tr1 = p1.get(key, []); vl1 = p1.get(f'val_{key}', [])
        tr2 = p2.get(key, []); vl2 = p2.get(f'val_{key}', [])
        if tr1: ax.plot(x1, tr1, 'b-o', ms=3, label='P1 Train')
        if vl1: ax.plot(x1, vl1, 'r-o', ms=3, label='P1 Val')
        if tr2: ax.plot(x2, tr2, 'b--o', ms=3, label='P2 Train')
        if vl2: ax.plot(x2, vl2, 'r--o', ms=3, label='P2 Val')
        if ep1 > 0:
            ax.axvline(x=ep1, color='gray', linestyle=':', alpha=0.7, label='P1→P2')
        if tgt is not None:
            ax.axhline(y=tgt, color='green', linestyle=':', alpha=0.7, label=tlbl)
        ax.set_title(title, fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    tc_path = os.path.join(PROJECT_DIR, 'training_curves.png')
    plt.savefig(tc_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {tc_path}')

No history file. Skipping training curves.


## 20. CBAM Attention Map Visualisation

In [23]:
import cv2

# Build attention map sub-model using CBAM layer directly
# Avoids get_layer('efficientnetb2') which crashes on flat saved model
cbam_layer = eval_model.get_layer('cbam')
cbam_input_tensor = cbam_layer.input

# --- FIXED: raw tf.* ops must be wrapped in a Lambda layer for Keras 3 / TF 2.16+ ---
ch_out = cbam_layer.channel_att(cbam_input_tensor)

combined = layers.Lambda(
    lambda t: tf.concat(
        [tf.reduce_mean(t, axis=-1, keepdims=True),
         tf.reduce_max(t,  axis=-1, keepdims=True)],
        axis=-1
    ),
    name='avg_max_concat'
)(ch_out)

spatial_mask = cbam_layer.spatial_att.conv(combined)

att_model = tf.keras.Model(
    inputs=eval_model.input,
    outputs=spatial_mask,
    name='attention_map_model'
)
print('Attention map model ready.')


def show_attention_maps(dataset, n=6):
    for batch_imgs, batch_labels in dataset.take(1):
        imgs   = batch_imgs.numpy()[:n]
        labels = batch_labels.numpy()[:n]
        break
    maps   = att_model.predict(imgs, verbose=0)
    preds  = eval_model.predict(imgs, verbose=0).flatten()

    fig, axes = plt.subplots(2, n, figsize=(3*n, 6))
    fig.suptitle('CBAM Spatial Attention — DeepSkin', fontsize=13, fontweight='bold')

    for i in range(n):
        img_u8 = np.clip(imgs[i], 0, 255).astype(np.uint8)
        t_lbl  = 'Malignant' if labels[i] == 1 else 'Benign'
        p_lbl  = 'Mal' if preds[i] >= 0.5 else 'Ben'
        axes[0, i].imshow(img_u8)
        axes[0, i].set_title(f'True:{t_lbl}\nPred:{p_lbl}({preds[i]:.2f})', fontsize=7)
        axes[0, i].axis('off')

        mask = maps[i, :, :, 0]
        mask = cv2.resize(mask, (260, 260))
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        hm   = cv2.applyColorMap((mask*255).astype(np.uint8), cv2.COLORMAP_JET)
        hm   = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB)
        ov   = cv2.addWeighted(img_u8, 0.6, hm, 0.4, 0)
        axes[1, i].imshow(ov)
        axes[1, i].set_title('Attention', fontsize=7)
        axes[1, i].axis('off')

    plt.tight_layout()
    att_path = os.path.join(PROJECT_DIR, 'attention_maps.png')
    plt.savefig(att_path, dpi=150); plt.show()
    print(f'Saved: {att_path}')


show_attention_maps(val_dataset, n=6)

Attention map model ready.


W0000 00:00:1783268583.758439  374168 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 207668224 bytes after encountering the first element of size 207668224 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
I0000 00:00:1783268585.540761  374235 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_302645__.28
I0000 00:00:1783268594.113384  374235 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_307236__.29


Saved: /home/higainai/project/deepskin_v4_gpu_optimization/DeepSkin_Project_CBAM/attention_maps.png


## 21. Final Summary

In [25]:
avg_prec_summary = average_precision_score(y_true, y_pred_proba)

total_samples = total_train + total_val
num_benign    = num_benign_train + num_benign_val
num_malignant = num_malignant_train + num_malignant_val

print('=' * 58)
print('     DeepSkin + CBAM  FINAL SUMMARY')
print('=' * 58)
print(f'Model            : EfficientNetB2 + CBAM')
print(f'Dataset          : {total_samples:,} images  ({num_benign:,} benign / {num_malignant:,} malignant)')
print(f'')
print(f'--- At threshold = 0.50 ---')
print(f'Recall           : {recall:.4f}  (target >0.90)')
print(f'Precision        : {precision:.4f}  (target >0.85)')
print(f'F1-Score         : {f1:.4f}  (target >0.87)')
print(f'F2-Score         : {f2:.4f}')
print(f'AUC-ROC          : {auc_roc_val:.4f}  (target >0.95)')
print(f'AUC-PR           : {avg_prec_summary:.4f}')
print(f'G-Mean           : {gmean:.4f}')
print(f'FNR              : {fnr_val:.4f}  (target <0.10)')
print(f'')
print(f'--- Best thresholds ---')
print(f'Best F1 thresh   : {best_f1_thresh:.2f}  (F1={best_f1:.4f})')
print(f'Best F2 thresh   : {best_f2_thresh:.2f}  (F2={best_f2:.4f})')
if thresh90:
    print(f'Recall>=0.90 thresh: {thresh90:.4f}')
print(f'')
print(f'Targets met      : {passed}/{len(all_metrics)}')
print(f'TP={tp}  TN={tn}  FP={fp}  FN={fn}')
print('=' * 58)
print(f'Outputs in: {PROJECT_DIR}')

     DeepSkin + CBAM  FINAL SUMMARY
Model            : EfficientNetB2 + CBAM
Dataset          : 25,331 images  (15,991 benign / 9,340 malignant)

--- At threshold = 0.50 ---
Recall           : 0.7714  (target >0.90)
Precision        : 0.7081  (target >0.85)
F1-Score         : 0.7384  (target >0.87)
F2-Score         : 0.7579
AUC-ROC          : 0.8797  (target >0.95)
AUC-PR           : 0.8185
G-Mean           : 0.7926
FNR              : 0.2286  (target <0.10)

--- Best thresholds ---
Best F1 thresh   : 0.45  (F1=0.7422)
Best F2 thresh   : 0.20  (F2=0.8221)
Recall>=0.90 thresh: 0.2974

Targets met      : 0/10
TP=1441  TN=2605  FP=594  FN=427
Outputs in: /home/higainai/project/deepskin_v4_gpu_optimization/DeepSkin_Project_CBAM


## 22. List Output Files

After training, download files from the **Output** tab on the right sidebar.

In [26]:
print(f'Output files in {PROJECT_DIR}:')
for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    if level > 0:
        print(f'{indent}{folder}/')
    for file in files:
        size_mb = os.path.getsize(os.path.join(root, file)) / 1e6
        print(f'{indent}  {file}  ({size_mb:.1f} MB)')

Output files in /home/higainai/project/deepskin_v4_gpu_optimization/DeepSkin_Project_CBAM:
  roc_curve.png  (0.1 MB)
  pr_curve.png  (0.1 MB)
  phase1_complete_model.keras  (37.4 MB)
  final_model_cbam.keras  (39.6 MB)
  confusion_matrices.png  (0.1 MB)
  attention_maps.png  (1.4 MB)
  checkpoints/
    training_metadata.json  (0.0 MB)
    efficientnet_b2_cbam_best_latest.keras  (39.6 MB)
    training_metadata_latest.json  (0.0 MB)
    efficientnet_b2_cbam_best.keras  (39.6 MB)


## 23. [Improvement] Day 1 — Test-Time Augmentation (TTA)

Runs each validation image through the model multiple times (original + horizontal flip + vertical flip + small zoom crops) and averages the sigmoid outputs. No retraining required — this only changes inference. Targets the domain-shift finding specifically, since TTA tends to stabilize predictions for borderline/uncertain cases near the decision threshold.


In [27]:
import tensorflow as tf
import numpy as np

def tta_predict(model, directory, class_names, img_size, n_tta=4, batch_size=BATCH_SIZE):
    """
    Test-Time Augmentation: averages predictions over the original image plus
    horizontal flip, vertical flip, and a zoom-in crop. Rebuilds a fresh,
    non-shuffling tf.data pipeline from the given directory.
    """
    base_ds = tf.keras.utils.image_dataset_from_directory(
        directory, labels='inferred', label_mode='binary',
        class_names=class_names, image_size=img_size,
        batch_size=batch_size, shuffle=False
    ).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=tf.data.AUTOTUNE) \
     .prefetch(tf.data.AUTOTUNE)

    y_true_tta = np.concatenate([y.numpy() for _, y in base_ds]).flatten().astype(int)

    augment_fns = [
        lambda x: x,                                     # original
        lambda x: tf.image.flip_left_right(x),            # horizontal flip
        lambda x: tf.image.flip_up_down(x),                # vertical flip
        lambda x: tf.image.central_crop(x, 0.85),          # mild zoom-in (needs resize back)
    ][:n_tta]

    all_preds = []
    for aug_fn in augment_fns:
        preds = []
        for batch_x, _ in base_ds:
            batch_aug = aug_fn(batch_x)
            if batch_aug.shape[1] != img_size[0] or batch_aug.shape[2] != img_size[1]:
                batch_aug = tf.image.resize(batch_aug, img_size)
            p = model.predict(batch_aug, verbose=0).flatten()
            preds.append(p)
        all_preds.append(np.concatenate(preds))

    y_pred_proba_tta = np.mean(all_preds, axis=0)
    return y_true_tta, y_pred_proba_tta


print('Running TTA on validation set (this takes ~n_tta times longer than a single pass)...')
y_true_tta, y_pred_proba_tta = tta_predict(
    eval_model, val_dir, class_names, IMG_SIZE, n_tta=4
)

# Compare against the non-TTA baseline already computed in Section 15/16
for thresh in [0.3, 0.4, 0.5]:
    y_p  = (y_pred_proba >= thresh).astype(int)
    y_pt = (y_pred_proba_tta >= thresh).astype(int)
    print(f"\nThreshold {thresh}:")
    print(f"  No TTA : recall={recall_score(y_true, y_p, zero_division=0):.4f}  "
          f"precision={precision_score(y_true, y_p, zero_division=0):.4f}  "
          f"F2={fbeta_score(y_true, y_p, beta=2, zero_division=0):.4f}")
    print(f"  +TTA   : recall={recall_score(y_true_tta, y_pt, zero_division=0):.4f}  "
          f"precision={precision_score(y_true_tta, y_pt, zero_division=0):.4f}  "
          f"F2={fbeta_score(y_true_tta, y_pt, beta=2, zero_division=0):.4f}")


Running TTA on validation set (this takes ~n_tta times longer than a single pass)...
Found 5067 files belonging to 2 classes.


W0000 00:00:1783268748.473960  381511 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 268436480 bytes after encountering the first element of size 268436480 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
W0000 00:00:1783268748.474120  374168 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 268436480 bytes after encountering the first element of size 268436480 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
W0000 00:00:1783268749.089559  374168 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 268436480 bytes after encountering the first element of size 268436480 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
I0000 00:00:1783268751.980876  

I0000 00:00:1783268760.396238  374233 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_312146__.29
W0000 00:00:1783268765.208209  382798 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 268436480 bytes after encountering the first element of size 268436480 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
W0000 00:00:1783268765.208560  374168 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 268436480 bytes after encountering the first element of size 268436480 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size
W0000 00:00:1783268768.424440  383295 prefetch_autotuner.cc:55] Prefetch autotuner tried to allocate 268436480 bytes after encountering the first element of size 268436480 bytes.This already causes the autotune ram budget to be


Threshold 0.3:
  No TTA : recall=0.8999  precision=0.5997  F2=0.8180
  +TTA   : recall=0.9154  precision=0.5952  F2=0.8265

Threshold 0.4:
  No TTA : recall=0.8464  precision=0.6604  F2=0.8012
  +TTA   : recall=0.8630  precision=0.6617  F2=0.8135

Threshold 0.5:
  No TTA : recall=0.7714  precision=0.7081  F2=0.7579
  +TTA   : recall=0.7907  precision=0.7098  F2=0.7731


## 24. [Improvement] Day 1 — Fine-Grained Threshold Sweep (Clinical Target)

The existing Section 17 sweep steps by 0.05 and optimizes F1/F2. This one steps by 0.02 and lets you pick a threshold by a stated clinical target (e.g. \"recall >= 0.90\") rather than by F-score alone, using the TTA predictions from Section 23.


In [28]:

TARGET_RECALL = 0.90   # <-- set your clinical minimum-acceptable recall here

print(f"{'Threshold':>10} {'Recall':>8} {'Precision':>10} {'F2':>8}")
print('-' * 42)
candidates = []
for thresh in np.arange(0.05, 0.61, 0.02):
    y_t = (y_pred_proba_tta >= thresh).astype(int)
    if y_t.sum() == 0:
        continue
    r = recall_score(y_true_tta, y_t, zero_division=0)
    p = precision_score(y_true_tta, y_t, zero_division=0)
    f2t = fbeta_score(y_true_tta, y_t, beta=2, zero_division=0)
    print(f"{thresh:>10.2f} {r:>8.4f} {p:>10.4f} {f2t:>8.4f}")
    if r >= TARGET_RECALL:
        candidates.append((thresh, r, p, f2t))

if candidates:
    # Highest threshold that still meets the recall target -> best precision at that target
    chosen = max(candidates, key=lambda c: c[0])
    OPERATING_THRESHOLD = chosen[0]
    print(f"\nChosen operating threshold = {OPERATING_THRESHOLD:.2f} "
          f"(recall={chosen[1]:.4f}, precision={chosen[2]:.4f}) "
          f"-- highest threshold that still meets target recall >= {TARGET_RECALL}")
else:
    OPERATING_THRESHOLD = 0.3
    print(f"\nNo threshold met target recall >= {TARGET_RECALL}; "
          f"defaulting OPERATING_THRESHOLD={OPERATING_THRESHOLD}. Consider Day 3-4 retrain experiments.")


 Threshold   Recall  Precision       F2
------------------------------------------
      0.05   0.9963     0.4353   0.7921
      0.07   0.9930     0.4461   0.7975
      0.09   0.9898     0.4571   0.8027
      0.11   0.9855     0.4696   0.8080
      0.13   0.9823     0.4818   0.8133
      0.15   0.9775     0.4931   0.8170
      0.17   0.9695     0.5067   0.8198
      0.19   0.9636     0.5186   0.8224
      0.21   0.9545     0.5338   0.8245
      0.23   0.9459     0.5471   0.8255
      0.25   0.9395     0.5611   0.8278
      0.27   0.9320     0.5757   0.8294
      0.29   0.9208     0.5892   0.8276
      0.31   0.9074     0.5996   0.8229
      0.33   0.8983     0.6122   0.8215
      0.35   0.8881     0.6279   0.8202
      0.37   0.8796     0.6420   0.8190
      0.39   0.8694     0.6559   0.8162
      0.41   0.8560     0.6682   0.8104
      0.43   0.8399     0.6769   0.8013
      0.45   0.8266     0.6850   0.7937
      0.47   0.8158     0.6949   0.7884
      0.49   0.8025     0.7064   0.78

## 25. [Improvement] Day 2 — Confidence Calibration (Temperature Scaling)

Fits a single scalar temperature T on the validation logits to make the sigmoid output a more reliable probability estimate (a calibrated 0.6 should mean ~60% of such predictions are truly positive). Also defines an 'uncertain' rejection band around the operating threshold for flagging low-confidence predictions for manual review.


In [29]:

from scipy.optimize import minimize_scalar
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def logit(p, eps=1e-7):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def nll_temperature(T, probs, labels):
    logits = logit(probs) / T
    p_scaled = sigmoid(logits)
    p_scaled = np.clip(p_scaled, 1e-7, 1 - 1e-7)
    return -np.mean(labels * np.log(p_scaled) + (1 - labels) * np.log(1 - p_scaled))

result = minimize_scalar(
    nll_temperature, bounds=(0.05, 5.0), method='bounded',
    args=(y_pred_proba_tta, y_true_tta)
)
TEMPERATURE = result.x
print(f"Fitted temperature: {TEMPERATURE:.4f}")

y_pred_proba_calibrated = sigmoid(logit(y_pred_proba_tta) / TEMPERATURE)

# Expected Calibration Error (ECE), before vs after
def expected_calibration_error(probs, labels, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_acc = labels[mask].mean()
        bin_conf = probs[mask].mean()
        ece += (mask.sum() / len(probs)) * abs(bin_acc - bin_conf)
    return ece

ece_before = expected_calibration_error(y_pred_proba_tta, y_true_tta)
ece_after  = expected_calibration_error(y_pred_proba_calibrated, y_true_tta)
print(f"ECE before calibration: {ece_before:.4f}")
print(f"ECE after calibration:  {ece_after:.4f}")

# ── Uncertain / rejection band around the chosen operating threshold ──
BAND_WIDTH = 0.10   # +/- around OPERATING_THRESHOLD counted as "uncertain"
lower = max(0.0, OPERATING_THRESHOLD - BAND_WIDTH)
upper = min(1.0, OPERATING_THRESHOLD + BAND_WIDTH)

def classify_with_rejection(probs, low=lower, high=upper):
    labels = np.where(probs < low, 'benign',
              np.where(probs > high, 'malignant', 'uncertain_refer'))
    return labels

decisions = classify_with_rejection(y_pred_proba_calibrated)
n_uncertain = np.sum(decisions == 'uncertain_refer')
print(f"\nUncertain band: [{lower:.2f}, {upper:.2f}]")
print(f"Flagged as 'uncertain -> refer for review': {n_uncertain} / {len(decisions)} "
      f"({n_uncertain/len(decisions)*100:.1f}%)")

# Accuracy on the confidently-decided subset only
confident_mask = decisions != 'uncertain_refer'
if confident_mask.sum() > 0:
    confident_pred = (y_pred_proba_calibrated[confident_mask] >= OPERATING_THRESHOLD).astype(int)
    print(f"Recall on confident-only subset: "
          f"{recall_score(y_true_tta[confident_mask], confident_pred, zero_division=0):.4f}")
    print(f"Precision on confident-only subset: "
          f"{precision_score(y_true_tta[confident_mask], confident_pred, zero_division=0):.4f}")


Fitted temperature: 0.9683
ECE before calibration: 0.0617
ECE after calibration:  0.0611

Uncertain band: [0.21, 0.41]
Flagged as 'uncertain -> refer for review': 915 / 5067 (18.1%)
Recall on confident-only subset: 0.9466
Precision on confident-only subset: 0.6693


## 26. [Improvement] Day 2 — Formalized Checkpoint Averaging

Formalizes the epoch-40 + epoch-50 combination as probability averaging across two saved checkpoints from Phase 2. Requires both checkpoints to exist on disk — if you only kept the 'best' and 'latest' checkpoints, this compares those two instead (documented explicitly either way).


In [30]:

# Point these at whichever two Phase-2 checkpoints you saved.
# Defaults to best vs latest, since those are guaranteed to exist under the current checkpointing scheme.
CKPT_A_PATH = os.path.join(CHECKPOINT_DIR, 'efficientnet_b2_cbam_best.keras')
CKPT_B_PATH = os.path.join(CHECKPOINT_DIR, 'efficientnet_b2_cbam_best_latest.keras')

model_a = tf.keras.models.load_model(CKPT_A_PATH, custom_objects=CUSTOM_OBJECTS, compile=False)
model_b = tf.keras.models.load_model(CKPT_B_PATH, custom_objects=CUSTOM_OBJECTS, compile=False)

# val_dataset re-iterates cleanly on each predict() call -- no .reset() needed
proba_a = model_a.predict(val_dataset, verbose=0).flatten()
proba_b = model_b.predict(val_dataset, verbose=0).flatten()

# Probability averaging (simpler and more common than logit averaging for a 2-snapshot ensemble)
proba_ensemble = (proba_a + proba_b) / 2.0

for name, proba in [('Checkpoint A only', proba_a), ('Checkpoint B only', proba_b),
                     ('Ensemble (avg probability)', proba_ensemble)]:
    y_p = (proba >= OPERATING_THRESHOLD).astype(int)
    print(f"{name:28s}  recall={recall_score(y_true, y_p, zero_division=0):.4f}  "
          f"precision={precision_score(y_true, y_p, zero_division=0):.4f}  "
          f"F2={fbeta_score(y_true, y_p, beta=2, zero_division=0):.4f}  "
          f"AUC-PR={average_precision_score(y_true, proba):.4f}")


/home/higainai/project/venv/lib/python3.12/site-packages/keras/src/layers/layer.py:431: UserWarning: `build()` was called on layer 'cbam', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
I0000 00:00:1783268897.928572  374233 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_334624__.29
I0000 00:00:1783268903.256794  374231 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_339458__.29
I0000 00:00:1783268907.396232  374231 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_344128__.29
I0000 00:00:1783268912.720105  374234 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_348962__.29


Checkpoint A only             recall=0.8967  precision=0.6089  F2=0.8192  AUC-PR=0.8185
Checkpoint B only             recall=0.8967  precision=0.6062  F2=0.8183  AUC-PR=0.8187
Ensemble (avg probability)    recall=0.8978  precision=0.6085  F2=0.8198  AUC-PR=0.8186


## 27. [Improvement] Day 3–4 — Phase 2 Variant Experiments

Resumes from the Phase 1 checkpoint (`phase1_complete_model.keras`) with two variables you can toggle: loss function (BCE vs Focal Loss) and unfreeze scope (block7 only vs block6+block7). Each variant saves to its own subfolder under `PROJECT_DIR/variants/` so it never overwrites your existing baseline checkpoints. Run this cell once per variant (change the config, rerun).


In [32]:

# ── Variant configuration -- change these two lines per experiment run ──
VARIANT_NAME   = "focal_block7"     # e.g. "bce_block7" (baseline), "focal_block7", "focal_block6_7"
USE_FOCAL_LOSS = True                # False = BinaryCrossentropy (the original baseline)
EXTRA_UNFREEZE_BLOCK6 = False         # True = also unfreeze block6 SE gates (larger capacity bump)

VARIANT_DIR = os.path.join(PROJECT_DIR, 'variants', VARIANT_NAME)
os.makedirs(VARIANT_DIR, exist_ok=True)
VARIANT_MODEL_FILE    = os.path.join(VARIANT_DIR, 'model_best.keras')
VARIANT_METADATA_FILE = os.path.join(VARIANT_DIR, 'metadata.json')

phase1_backup = os.path.join(PROJECT_DIR, 'phase1_complete_model.keras')
model = tf.keras.models.load_model(phase1_backup, custom_objects=CUSTOM_OBJECTS, compile=False)
print(f"Loaded Phase 1 backup for variant: {VARIANT_NAME}")

for layer in model.layers:
    layer.trainable = False

UNFREEZE_NAMES = {
    'block7b_dwconv', 'block7b_se_reduce', 'block7b_se_expand',
    'block7a_dwconv', 'block7a_se_reduce', 'block7a_se_expand',
    'cbam', 'gap', 'bn_head', 'dropout_head', 'output',
}
if EXTRA_UNFREEZE_BLOCK6:
    UNFREEZE_NAMES |= {
        'block6a_dwconv', 'block6a_se_reduce', 'block6a_se_expand',
        'block6b_dwconv', 'block6b_se_reduce', 'block6b_se_expand',
        'block6c_dwconv', 'block6c_se_reduce', 'block6c_se_expand',
        'block6d_dwconv', 'block6d_se_reduce', 'block6d_se_expand',
    }
    # NOTE: verify these exact layer names exist in your EfficientNetB2 graph
    # (model.summary() or [l.name for l in model.layers if "block6" in l.name])
    # before relying on this -- EfficientNetB2 has more block6 sub-blocks than block7.

for layer in model.layers:
    if layer.name in UNFREEZE_NAMES:
        layer.trainable = True
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Trainable params: {trainable_params:,}")
if trainable_params > 2_000_000:
    raise ValueError(f"Trainable param count ({trainable_params:,}) looks too high -- check UNFREEZE_NAMES.")

loss_fn = BinaryFocalLoss(alpha=0.35, gamma=2.0) if USE_FOCAL_LOSS else 'binary_crossentropy'
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss=loss_fn, metrics=tracking_metrics,
              jit_compile=True)   # XLA: fuses ops, reduces memory-transfer overhead

variant_ckpt = FullModelCheckpoint(VARIANT_MODEL_FILE, VARIANT_METADATA_FILE, 'phase2_variant', 0)

history_variant = model.fit(
    train_dataset, validation_data=val_dataset,
    epochs=40, initial_epoch=0,
    class_weight=None if USE_FOCAL_LOSS else class_weight_dict,   # CHANGED: avoid double-correcting imbalance when focal loss's alpha already does it
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_auc_pr', patience=8, mode='max', restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_auc_pr', factor=0.5, patience=4, mode='max'),
        variant_ckpt, real_recall_cb,
    ]
)

model.save(os.path.join(VARIANT_DIR, 'model_final.keras'))
best_val_auc_pr = float(max(history_variant.history.get('val_auc_pr', [0])))
print(f"\nVariant '{VARIANT_NAME}' complete. Best val_auc_pr = {best_val_auc_pr:.4f}")
print(f"Compare this against your baseline val_auc_pr = 0.8192 to see if this variant helps.")


/home/higainai/project/venv/lib/python3.12/site-packages/keras/src/layers/layer.py:431: UserWarning: `build()` was called on layer 'cbam', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Loaded Phase 1 backup for variant: focal_block7
Trainable params: 784,559
Epoch 1/40


/home/higainai/project/venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1783269414.857318  374231 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_480757__.376


79/80 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.7951 - auc_pr: 0.7959 - auc_roc: 0.8689 - loss: 0.0610 - precision: 0.7781 - recall: 0.6156

I0000 00:00:1783269440.600134  374231 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_480757__.376


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step - accuracy: 0.7952 - auc_pr: 0.7960 - auc_roc: 0.8689 - loss: 0.0609 - precision: 0.7783 - recall: 0.6156

I0000 00:00:1783269451.075914  374231 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'fusion_531', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1783269453.557451  374234 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_486980__.50
I0000 00:00:1783269458.182781  374231 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_486980__.50


  ✓ Best saved — epoch 1, val_auc_pr=0.7977


I0000 00:00:1783269463.813442  374231 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_495619__.29
I0000 00:00:1783269469.193623  374235 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_500453__.29



  [RealRecall] Epoch 1:
    thresh=0.3: recall=0.8988  precision=0.5838  F2=0.8113
    thresh=0.4: recall=0.7912  precision=0.6808  F2=0.7664
    thresh=0.5: recall=0.6370  precision=0.7638  F2=0.6589
80/80 ━━━━━━━━━━━━━━━━━━━━ 68s 596ms/step - accuracy: 0.7952 - auc_pr: 0.7960 - auc_roc: 0.8689 - loss: 0.0609 - precision: 0.7783 - recall: 0.6156 - val_accuracy: 0.7936 - val_auc_pr: 0.7977 - val_auc_roc: 0.8697 - val_loss: 0.0575 - val_precision: 0.7638 - val_recall: 0.6370 - learning_rate: 1.0000e-05
Epoch 2/40
79/80 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step - accuracy: 0.7884 - auc_pr: 0.7966 - auc_roc: 0.8694 - loss: 0.0555 - precision: 0.7795 - recall: 0.5943  No improvement (best=0.7977)

  [RealRecall] Epoch 2:
    thresh=0.3: recall=0.9197  precision=0.5635  F2=0.8165
    thresh=0.4: recall=0.8025  precision=0.6740  F2=0.7730
    thresh=0.5: recall=0.6097  precision=0.7801  F2=0.6376
80/80 ━━━━━━━━━━━━━━━━━━━━ 19s 216ms/step - accuracy: 0.7882 - auc_pr: 0.7965 - auc_roc: 0.8693 - loss

## 28. [Improvement] Day 5 — Multi-Source Validation & Augmentation Tuning (Manual Steps)

These two items need new data / a config rerun rather than a pure code addition here:

**Multi-source validation (recommended, addresses the domain-shift finding directly):**
1. Download an independent ISIC batch that you can *confirm* has zero overlap with training (run `overlap_check.py` against it first).
2. Hold this out as a **third, separate test set** — distinct from your current `val/` split — so you get a genuine, never-touched generalization number, not just another slice of the same combined pool.
3. Re-run Section 15-18's evaluation cells against this set using `flow_from_directory` pointed at the new folder in place of `validation_generator`.

**Augmentation tuning for domain robustness:**
1. In the Section 9-ish data-generator cell, add stronger color/contrast jitter to `train_datagen` -- e.g. `channel_shift_range=20.0` and a wider `brightness_range=[0.7, 1.3]` -- to simulate different camera/lighting pipelines.
2. Re-run a Phase 2 variant (Section 27) with this changed generator, and compare its performance specifically on the independent multi-source test set from step above, not just your existing val split -- that's the number that tells you whether this actually helped generalization.


## 29. [Improvement] Day 6 — Combined Final Evaluation

Brings together whichever combination of (best Phase-2 variant) + (checkpoint averaging) + (TTA) + (calibration) + (chosen operating threshold) performed best across Days 1-5, evaluated on ALL three sets: local val, clean ISIC subset, and the new multi-source held-out set from Day 5. Fill in MODEL_PATHS_TO_ENSEMBLE and re-run Sections 23-26 against each test set in turn by swapping `validation_generator` for the relevant generator, then tabulate all results here for the defense document.


In [ ]:

# Template for the final comparison table -- fill in after running Days 1-5 across all test sets.
final_comparison = [
    # (config_name,                    dataset,        recall, precision, f2, auc_pr)
    ("Baseline (BCE, block7, thresh=0.5)",       "Local val",      None, None, None, None),
    ("Baseline (BCE, block7, thresh=0.5)",       "Clean ISIC",     None, None, None, None),
    ("+ Best threshold",                          "Local val",      None, None, None, None),
    ("+ Best threshold",                          "Clean ISIC",     None, None, None, None),
    ("+ TTA",                                     "Local val",      None, None, None, None),
    ("+ TTA",                                     "Clean ISIC",     None, None, None, None),
    ("+ Checkpoint ensemble",                     "Local val",      None, None, None, None),
    ("+ Checkpoint ensemble",                     "Clean ISIC",     None, None, None, None),
    ("+ Calibration",                             "Local val",      None, None, None, None),
    ("+ Calibration",                             "Clean ISIC",     None, None, None, None),
    ("Best Phase-2 variant (fill in name)",       "Local val",      None, None, None, None),
    ("Best Phase-2 variant (fill in name)",       "Clean ISIC",     None, None, None, None),
    ("Best Phase-2 variant (fill in name)",       "New multi-source", None, None, None, None),
]

import pandas as pd
df_final = pd.DataFrame(final_comparison,
    columns=["Configuration", "Dataset", "Recall", "Precision", "F2", "AUC-PR"])
print(df_final.to_string(index=False))
print("\nFill in the None values as each Day 1-5 cell is run against each dataset,")
print("then this table drops straight into the defense document's results section.")


                      Configuration          Dataset Recall Precision   F2 AUC-PR
 Baseline (BCE, block7, thresh=0.5)        Local val   None      None None   None
 Baseline (BCE, block7, thresh=0.5)       Clean ISIC   None      None None   None
                   + Best threshold        Local val   None      None None   None
                   + Best threshold       Clean ISIC   None      None None   None
                              + TTA        Local val   None      None None   None
                              + TTA       Clean ISIC   None      None None   None
              + Checkpoint ensemble        Local val   None      None None   None
              + Checkpoint ensemble       Clean ISIC   None      None None   None
                      + Calibration        Local val   None      None None   None
                      + Calibration       Clean ISIC   None      None None   None
Best Phase-2 variant (fill in name)        Local val   None      None None   None
Best Phase-2 var

: 